# 13. Debugging · Quality · Config

문제를 빠르게 찾고, 문서 품질을 자동 점검하고, App 기본 설정을 공유하는 도구.

이 튜토리얼의 모든 출력은 **실제 HWP 실행 결과**입니다 (hidden cell 이 HWP
subprocess 를 호출해 캡처).


## 1. Discovery — `app.help()` / `repr(app)`

```python
app = App()
app.help()       # 18개 accessor + context manager 출력
repr(app)        # 상태 요약
```

실제 출력:


```
App(visible=False, version='12, 0, 0, 3146', docs=1, page=1/1)

════════════════════════════════════════════════════════════════════
 hwpapi · App 사용 가능 API
════════════════════════════════════════════════════════════════════

· Navigation & Selection
    app.move              — 커서 이동 (.doc.top(), .line.end() ...)
    app.sel               — 현재 선택 제어 (.current_paragraph(), .compress())

· Collections
    app.documents         — 열린 문서 컬렉션
    app.fields            — 누름틀(필드) — dict/list-like
    app.bookmarks         — 책갈피 — .add(), .goto(), in
    app.hyperlinks        — 하이퍼링크 — .add(text, url)
    app.images            — 이미지 컬렉션 — .resize_all(), .grayscale_all()
    app.styles            — 문단 스타일 — .apply(name)
    app.controls          — 문서 내 모든 control 순회

· Structure
    app.cell              — 표 셀 조작
    app.table             — 표 구조 + 일괄 서식 — .header_row(), .align()
    app.page              — 페이지 속성

· Transform & View
    app.convert           — 숫자→한글, 폰트 교체, 줄 나눔
    app.view              — zoom, 전체화면, 페이지 모드

· Quality & Templates
    app.lint              — 문서 품질 체크 (callable → LintReport)
    app.template          — 템플릿 .save() / .apply()
    app.config            — App 기본 선호도

· Presets & Debug
    app.preset            — 꾸미기 프리셋 (title_box, striped_rows, toc …)
    app.debug             — .state(), .trace(), .timing()

· Context managers (with 구문으로 사용)
    with app.silenced(mode='yes')           — 다이얼로그 자동 응답 (종료 시 복원)
    with app.suppress_errors()              — 에러 dialog ABORT + Python 예외 swallow
    with app.batch_mode(hide=True)          — 창 숨김 + dialog 억제 — 대량 처리 5~10배 가속
    with app.undo_group(description)        — 블록 내 편집을 단일 undo 경계로
    with app.charshape_scope(**fmt)         — 블록 내 문자 모양 임시 변경
    with app.parashape_scope(**fmt)         — 블록 내 문단 모양 임시 변경
    with app.use_document(doc)              — 블록 내 활성 문서 일시 전환
    with app.debug.trace(verbose=True)      — 블록 내 COM Run() 호출 로그

· 주요 property
    app.text
    app.visible
    app.version
    app.page_count
    app.current_page
    app.selection
    app.charshape
    app.parashape
    app.status

' app.help()' 로 언제든 다시 확인할 수 있습니다.
════════════════════════════════════════════════════════════════════
```


## 2. app.debug

### 2.1 state() — 현재 상태 dict

```python
state = app.debug.state()
```

실제 반환 값:


```
{'charshape_summary': {'bold': 255,
                       'fontsize': 1100,
                       'italic': 0,
                       'shade_color': Color('#ffffff'),
                       'text_color': Color(0)},
 'cursor': {'doc_id': 0, 'para': 0, 'pos': 22},
 'documents_open': 2,
 'filepath': '(unsaved)',
 'in_table': False,
 'page': {'current': 1, 'total': 1},
 'selection': {'length': 0, 'text': ''},
 'version': '12, 0, 0, 3146',
 'visible': True}
```


### 2.2 print() — 상태 예쁘게 출력

```python
app.debug.print()
```

실제 출력:


```
============================================================
 hwpapi debug state @ 22:47:24
============================================================
  cursor:
    doc_id               = 0
    para                 = 1
    pos                  = 9
  page:
    current              = 2
    total                = 1
  selection:
    text                 = ''
    length               = 0
  charshape_summary:
    fontsize             = 1000
    bold                 = 0
    italic               = 0
    text_color           = Color(0)
    shade_color          = Color('#ffffff')
  in_table               = False
  documents_open         = 2
  visible                = True
  version                = '12, 0, 0, 3146'
  filepath               = '(unsaved)'
============================================================
```


### 2.3 timing(fn) — 함수 실행 시간 측정

```python
r = app.debug.timing(app.insert_text, "테스트 텍스트")
print(r)
```

실제 결과:


```
insert_text('측정 대상 텍스트'):
{'elapsed_ms': 3109.633,
 'exception': None,
 'result': App(visible=True, version='12, 0, 0, 3146', docs=2, page=1/1),
 'success': True}

page_count:
{'elapsed_ms': 0.998, 'exception': None, 'result': 1, 'success': True}

lint():
  elapsed_ms: 8.348
  success: True
```


### 2.4 trace() — COM Run() 호출 추적

```python
with app.debug.trace():
    app.preset.title_box(text="Test")
```

`#| echo: false` 로 hidden 된 블록의 실제 trace 출력:


```
[trace   0] Run('InsertText') → True (1142.98ms)
[trace   1] Run('SelectAll') → True (8.15ms)
[trace   2] Run('Cancel') → True (7.86ms)
[trace] total: 3 Run() calls
```


## 3. app.lint() — 문서 품질 체크

```python
report = app.lint()
print(report.summary)
```

실제 lint 결과 (샘플 문서 — 긴 문장 + 빈 문단 + 이중 공백 포함):


```
Document lint — 7 issue(s)
  Total: 6 paragraphs, 4 sentences, 132 chars
  Empty paragraphs: 2
  Double spaces: 1 para(s)
  Trailing whitespace: 4 para(s)

상세 — long_sentences: []
상세 — empty_paragraphs: [2, 5]
상세 — double_spaces: [3]
has_issues: True
issue_count: 7
```


*위 report 가 감지한 문서:*

![demo — lint_target](img/demo_lint_target.png)


## 4. app.template — 문서 템플릿

```python
app.template.save("company_style.json")
```

저장된 JSON 실제 내용:


```
{
  "format": "hwpapi-template",
  "version": "1.0",
  "charshape": {
    "fontsize": 1100,
    "bold": 255,
    "italic": 0,
    "text_color": 0,
    "facename_hangul": "함초롬바탕",
    "facename_latin": "함초롬바탕"
  },
  "parashape": {
    "align_type": 0,
    "line_spacing": 160
  },
  "page": {}
}
```


### template.apply — 다른 문서에 적용

```python
app2 = App()
app2.open("new_doc.hwp")
app2.template.apply("company_style.json")
# → charshape, parashape 일괄 반영
```


## 5. app.config — App 기본 선호도

```python
app.config.default_font = "함초롬바탕"
app.config.default_size = 11
app.config.palette = {
    "primary": "#003366",
    "accent": "#FF6600",
}
app.config.to_dict()
```

실제 출력:


In [ ]:
import pprint
from hwpapi.core import App
a = App(is_visible=False)

# 설정
a.config.default_font = "함초롬바탕"
a.config.default_size = 11
a.config.default_line_spacing = 160
a.config.palette = {
    "primary": "#003366",
    "accent": "#FF6600",
    "bg_alt": "#EEEEEE",
}

# dict 로
print(">>> app.config.to_dict()")
pprint.pprint(a.config.to_dict(), width=70)

# reset 동작
print("\n>>> app.config.reset()")
a.config.reset()
pprint.pprint(a.config.to_dict(), width=70)

try: a.quit()
except Exception: pass

### config 디스크 저장 / 로드

```python
app.config.save("~/.hwpapirc")       # 저장
app2.config.load("~/.hwpapirc")      # 다른 세션에서 로드
```

설정 파일 (JSON):


```
{
  "default_font": "함초롬바탕",
  "default_size": 11,
  "default_line_spacing": null,
  "default_table_style": null,
  "palette": {
    "primary": "#003366"
  }
}
```


## 6. 실전 디버그 세션

```python
app = App()
app.open("problem.hwp")

# 1. 상태 파악
app.debug.print()

# 2. 문서 품질 체크
report = app.lint()
if report.has_issues:
    print(report.summary)

# 3. 느린 작업 조사
for fn_name in ["current_page", "page_count", "lint"]:
    fn = getattr(app, fn_name)
    r = app.debug.timing(lambda: fn() if callable(fn) else fn)
    print(f"{fn_name:20s} {r['elapsed_ms']:.1f}ms")

# 4. 특정 작업 trace
with app.debug.trace():
    app.preset.striped_rows()
```

위 세션을 실제로 실행한 결과:


```
{'charshape_summary': {'bold': 255,
                       'fontsize': 1100,
                       'italic': 0,
                       'shade_color': Color('#ffffff'),
                       'text_color': Color(0)},
 'cursor': {'doc_id': 0, 'para': 0, 'pos': 22},
 'documents_open': 2,
 'filepath': '(unsaved)',
 'in_table': False,
 'page': {'current': 1, 'total': 1},
 'selection': {'length': 0, 'text': ''},
 'version': '12, 0, 0, 3146',
 'visible': True}
```
